In [1]:
import subprocess
import os

result = subprocess.run('bash -c "source /etc/network_turbo && env | grep proxy"', shell=True, capture_output=True, text=True)
output = result.stdout
for line in output.splitlines():
    if '=' in line:
        var, value = line.split('=', 1)
        os.environ[var] = value
os.environ['HF_HOME'] = "/root/autodl-tmp/.cache/huggingface"

# 导入必要的库

In [2]:
# 导入基础库
import torch
import numpy as np
from datasets import load_dataset, DatasetDict, Audio
from dataclasses import dataclass
from typing import Any, Dict, List, Union

# 导入 Transformers 库
from transformers import (
    AutoFeatureExtractor, AutoTokenizer, AutoProcessor, 
    AutoModelForSpeechSeq2Seq, Seq2SeqTrainingArguments, Seq2SeqTrainer
)

# 导入 PEFT 库
from peft import (
    LoraConfig, PeftModel, PeftConfig, get_peft_model, 
    prepare_model_for_kbit_training
)

# 配置参数

In [3]:
# 基础模型
MODEL_NAME = "openai/whisper-large-v2"

# 保存路径
OUTPUT_DIR = "models/whisper-large-v2-lora"

# LoRA 配置
LORA_RANK = 4         # LoRA 低秩矩阵的秩
LORA_ALPHA = 64       # LoRA 缩放因子
LORA_DROPOUT = 0.05   # LoRA dropout率

# 训练参数
BATCH_SIZE = 64       # 批次大小
LEARNING_RATE = 1e-3  # 学习率
NUM_EPOCHS = 1        # 训练轮数

# 语言配置
LANGUAGE = "Chinese (China)"
LANGUAGE_CODE = "zh-CN"
TASK = "transcribe"

# 数据集配置
DATASET_NAME = "mozilla-foundation/common_voice_11_0"
TRAIN_SAMPLES = 640   # 训练样本数（用于演示）
EVAL_SAMPLES = 320    # 验证样本数（用于演示）

# 数据准备

## 加载数据集

In [4]:
# 初始化数据集字典
common_voice = DatasetDict()

# 加载训练集和验证集
common_voice["train"] = load_dataset(DATASET_NAME, LANGUAGE_CODE, split="train", trust_remote_code=True)
common_voice["validation"] = load_dataset(DATASET_NAME, LANGUAGE_CODE, split="validation", trust_remote_code=True)

# 查看数据集样本
print(f"训练集: {len(common_voice['train'])} 样本")
print(f"验证集: {len(common_voice['validation'])} 样本")
print("\n样本示例:")
print(common_voice["train"][0])

RuntimeError: Data processing error: CAS service error : Error : single flight error: Real call failed: ReqwestMiddlewareError(Reqwest(reqwest::Error { kind: Request, url: "https://transfer.xethub.hf.co/xorbs/default/31d9b7757a485c4534600d9562dc6d31ef0226f729863a30488ab8893db63eee?X-Xet-Signed-Range=bytes%3D0-64317186&Expires=1747635658&Policy=eyJTdGF0ZW1lbnQiOlt7IlJlc291cmNlIjoiaHR0cHM6Ly90cmFuc2Zlci54ZXRodWIuaGYuY28veG9yYnMvZGVmYXVsdC8zMWQ5Yjc3NTdhNDg1YzQ1MzQ2MDBkOTU2MmRjNmQzMWVmMDIyNmY3Mjk4NjNhMzA0ODhhYjg4OTNkYjYzZWVlP1gtWGV0LVNpZ25lZC1SYW5nZT1ieXRlcyUzRDAtNjQzMTcxODYiLCJDb25kaXRpb24iOnsiRGF0ZUxlc3NUaGFuIjp7IkFXUzpFcG9jaFRpbWUiOjE3NDc2MzU2NTh9fX1dfQ__&Signature=TppTSNwX-Dz5xz26xMBOJX-dx4XsMyq1kzt1W6dilcqYcQovD~7UDtOxSDmfYYsjTizgNYVC-H0lJKJQNBMrmmOk2tK6DMWTHQnY34xlaCQTJRCF0eIeQjdW3NmTck2QYU-2OI7~i0w9xum5eyH9UhvsjBmKM-mVX5UjG3Mt61uKrdccjl7Qc2oo-RZeXBB-Irywou8o7nbggK8Tgdx~48j1BxLyw5XdnwB7j0vQ5boEiTzd~YC2GjCog~Vgo5VV4OURRpyxmV2ajXlSDuuephBadCPMId~670P6jDCVaBj0Ap1eP9~sUHe0Z128e9QMKkiFPJb

In [ ]:
from typing import Union
from datasets import Dataset, DatasetDict
import pandas as pd

def inspect_dataset(
    ds: Union[Dataset, DatasetDict],
    num_samples: int = 5,
    seed: int = 42
) -> None:
    """
    快速查看 Hugging Face Dataset 或 DatasetDict 的结构与示例。

    参数
    ----
    ds : Dataset 或 DatasetDict
        要检查的数据集（可以是单个 split 的 Dataset，也可以是多个 split 的 DatasetDict）。
    num_samples : int, optional (default=5)
        每个 split 中要随机抽看的示例条目数。
    seed : int, optional (default=42)
        随机种子，用于可复现地抽样示例。

    输出
    ----
    该函数会依次打印每个 split 的：
      1. split 名称（对于 Dataset 则视为 'default'）
      2. 总样本数
      3. 所有字段（columns）及其类型
      4. 随机抽取的若干行示例（以 pandas.DataFrame 形式展示）
    """
    def _inspect_split(name: str, dset: Dataset):
        print(f"\n--- Split: {name} ---")
        print(f"样本数：{len(dset)}")
        print("字段与类型:")
        for col, feat in dset.features.items():
            print(f"  - {col}: {feat}")
        # 随机抽样并展示
        sample_idx = dset.shuffle(seed=seed).select(range(min(num_samples, len(dset))))
        df = pd.DataFrame(sample_idx)
        display(df)   # 在 Notebook 中可直接展示；普通脚本可改成 print(df.head())

    # 如果是 DatasetDict，遍历每个 split
    if isinstance(ds, DatasetDict):
        for split_name, split_ds in ds.items():
            _inspect_split(split_name, split_ds)
    # 如果是单一的 Dataset
    else:
        _inspect_split("default", ds)

inspect_dataset(common_voice, num_samples=5)

## 加载预处理组件

In [ ]:
# 加载特征提取器、分词器和处理器
feature_extractor = AutoFeatureExtractor.from_pretrained(MODEL_NAME)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, language=LANGUAGE, task=TASK)
processor = AutoProcessor.from_pretrained(MODEL_NAME, language=LANGUAGE, task=TASK)

## 数据预处理

In [ ]:
# 移除不必要的列
common_voice = common_voice.remove_columns(
    ["accent", "age", "client_id", "down_votes", "gender", "locale", "path", "segment", "up_votes"]
)

# 将音频降采样到16kHz (Whisper的预训练采样率)
common_voice = common_voice.cast_column("audio", Audio(sampling_rate=16000))

# 创建数据预处理函数
def prepare_dataset(batch):
    """处理数据集样本: 提取音频特征并标记文本"""
    # 获取音频数据
    audio = batch["audio"]
    
    # 使用特征提取器处理音频
    batch["input_features"] = feature_extractor(
        audio["array"], 
        sampling_rate=audio["sampling_rate"]
    ).input_features[0]
    
    # 使用分词器处理文本
    batch["labels"] = tokenizer(batch["sentence"]).input_ids
    return batch

# 数据抽样 (演示用)
sampled_dataset = DatasetDict()
sampled_dataset["train"] = common_voice["train"].shuffle(seed=42).select(range(TRAIN_SAMPLES))
sampled_dataset["validation"] = common_voice["validation"].shuffle(seed=42).select(range(EVAL_SAMPLES))

# 应用预处理
tokenized_dataset = sampled_dataset.map(prepare_dataset)

print(f"处理后的数据集: {tokenized_dataset}")

## 创建数据整理器

In [ ]:
@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    """
    用于语音到文本模型的数据整理器: 处理批次中的填充和掩码
    """
    processor: Any
    
    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        # 提取和填充输入特征
        input_features = [{"input_features": feature["input_features"]} for feature in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")
        
        # 提取和填充标签
        label_features = [{"input_ids": feature["labels"]} for feature in features]
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")
        
        # 将填充标记替换为-100，以便在损失计算中忽略它们
        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)
        
        # 如果所有样本都以BOS标记开头，则删除它
        if (labels[:, 0] == self.processor.tokenizer.bos_token_id).all().cpu().item():
            labels = labels[:, 1:]
            
        batch["labels"] = labels
        return batch

# 创建数据整理器实例
data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor=processor)